!pip install pandas

In [2]:
!pip install pandas

zsh:1: command not found: pip


In [3]:
import pandas as pd
import sqlite3



In [6]:
conn = sqlite3.connect(':memory:') # Create temporary database
items= pd.read_csv('data/items.csv').to_sql('items', conn, index=False)

In [4]:
QUERY = "SELECT * FROM items"
pd.read_sql_query(QUERY, conn)

,item_id,description,dept,category,unit_of_measure,case_size,unit_cost
0,10001,Bananas,Produce,Tropical,LB,36,0.28
1,10002,Strawberries 1lb,Produce,Berries,LB,20,2.32
2,10003,Blueberries 6oz,Produce,Berries,LB,18,2.84
3,10004,Romaine Hearts 3ct,Produce,Salad,EA,18,3.34
4,10005,Baby Spinach 5oz,Produce,Salad,LB,24,1.93
...,...,...,...,...,...,...,...
195,10196,Coffee Ground 12oz,Grocery,Beverages,EA,48,0.81
196,10197,Tea Bags 40ct,Grocery,Beverages,EA,15,5.90
197,10198,Soda Cola 12pk,Grocery,Beverages,EA,30,4.39
198,10199,Sparkling Water 12pk,Grocery,Beverages,EA,40,2.44


In [7]:
sales_daily = pd.read_csv('data/sales_daily.csv').to_sql('sales_daily', conn, index=False)
shipments = pd.read_csv('data/shipments.csv').to_sql('shipments', conn, index=False)

In [8]:

conn.execute( '''          create view shrink_by_item_store_date as
select 
items.item_id,
shipments.store_id,
items.dept,
items.category,
shipments.date,
sum(shipments.cases_received*items.case_size) as units_shipped_on_date,
sum(sales_daily.units_sold) as units_sold_on_date,
sum(shipments.cases_received*items.case_size)- sum(sales_daily.units_sold) as shrink_on_date
from items
left join shipments on items.item_id = shipments.item_id
left join(
    select
    item_id,
     store_id,
     date,
     sum(units_sold) as units_sold
    from sales_daily
    group by item_id, store_id, date
)sales_daily on items.item_id = sales_daily.item_id and shipments.store_id = sales_daily.store_id and shipments.date = sales_daily.date
group by items.item_id, shipments.store_id, items.dept, items.category, shipments.date

''')

In [9]:
pd.read_sql_query( '''

create view shrink_by_item_store_date as
select 
items.item_id,
shipments.store_id,
items.dept,
items.category,
shipments.date,
sum(shipments.cases_received*items.case_size) as units_shipped_on_date,
sum(sales_daily.units_sold) as units_sold_on_date,
sum(shipments.cases_received*items.case_size)- sum(sales_daily.units_sold) as shrink_on_date
from items
left join shipments on items.item_id = shipments.item_id
left join(
    select
    item_id,
     store_id,
     date,
     sum(units_sold) as units_sold
    from sales_daily
    group by item_id, store_id, date
)sales_daily on items.item_id = sales_daily.item_id and shipments.store_id = sales_daily.store_id and shipments.date = sales_daily.date
group by items.item_id, shipments.store_id, items.dept, items.category, shipments.date

''',conn)

TypeError: 'NoneType' object is not iterable

In [ ]:
conn.execute('DROP VIEW IF EXISTS daily_facts')
conn.execute('''CREATE VIEW daily_facts AS
WITH ship AS (
    SELECT s.date, s.store_id, s.item_id, s.banner, s.region,
           cases_received * case_size             AS shipped_units,
           cases_received * case_size * unit_cost AS shipped_cost
    FROM shipments s JOIN items i USING (item_id)
),
sale AS (
    SELECT s.date, s.store_id, s.item_id, s.banner, s.region,
           units_sold             AS sold_units,
           units_sold * unit_cost AS sold_cost,
           s.net_sales
    FROM sales_daily s JOIN items i USING (item_id)
)
-- FULL OUTER JOIN, and never ON a shipment date. This builds a
-- zero-filled store-item-day spine so that SUM over any period or
-- grouping is correct, and so the rows that shipped and never sold
-- -- the highest-shrink rows in the extract -- are not dropped.
SELECT
    COALESCE(sh.date, sa.date)          AS date,
    COALESCE(sh.store_id, sa.store_id)  AS store_id,
    COALESCE(sh.item_id, sa.item_id)    AS item_id,
    COALESCE(sh.banner, sa.banner)      AS banner,
    COALESCE(sh.region, sa.region)      AS region,
    i.dept, i.category, i.description, i.unit_cost,
    COALESCE(sh.shipped_units, 0)       AS shipped_units,
    COALESCE(sh.shipped_cost, 0.0)      AS shipped_cost,
    COALESCE(sa.sold_units, 0)          AS sold_units,
    COALESCE(sa.sold_cost, 0.0)         AS sold_cost,
    COALESCE(sa.net_sales, 0.0)         AS net_sales
FROM ship sh
FULL OUTER JOIN sale sa
  ON sh.date = sa.date AND sh.store_id = sa.store_id
 AND sh.item_id = sa.item_id
JOIN items i ON i.item_id = COALESCE(sh.item_id, sa.item_id)''')

In [13]:
pd.read_sql_query( '''
select distinct * from daily_facts
''',conn)

DatabaseError: Execution failed on sql '
select distinct * from daily_facts
': no such column: sh.shipped_units

In [14]:
pd.read_sql_query( '''
select * from sales_daily
where item_id = 10001 and store_id = 101	
and date = '2026-04-01'
''',conn)

,date,store_id,banner,region,item_id,units_sold,net_sales
0,2026-04-01,101,Meridian Foods,Northeast,10001,51,20.91
